# VascuMap Batch Pipeline

Runs the full VascuMap pipeline on every image in a source folder.  
- **LIF files**: iterates through each image within the file  
- **TIF/TIFF files**: one pipeline run per file  

Set `record_timings = True` to save a per-image timing breakdown CSV alongside the outputs.

**Default outputs** (always saved):
| File | Description |
|------|-------------|
| `*_overlay_geometry_0.tif` | 2-D device overlay with inner/outer geometry |
| `*_skeleton_overview.png` | Skeleton overview plot |
| `*_analysis_metrics.csv` | Quantitative vascular network metrics |

**Extra outputs** (when `save_all_interim = True`):
| File | Description |
|------|-------------|
| `*_cropped_stack_aligned.npy` | Brightfield stack at 2 µm iso |
| `*_vessel_translation_aligned.npy` | Pix2Pix image translation |
| `*_clean_segmentation.npy` | Cleaned binary vessel segmentation |
| `*_skeleton.npy` | Skeleton from pruned graph |
| `*_holes.npy` | Binary pore mask |
| `*_hole_labels_per_slice.npy` | Per-slice labelled pores |
| `*_hole_distance_per_slice_um.npy` | Inscribed-radius distance map |
| `*_full_graph_skeleton.npy` | Pre-pruning skeleton |
| `*_vessel_mask.npy` | Raw vessel mask |
| `*_graph_nodes.npz` | Sprout/junction node coords |
| `*_clean_graph.pkl` | NetworkX graph (pickle) |

In [1]:
import sys, time
from pathlib import Path

sys.path.insert(0, str(Path(r"C:\Users\taylorhearn\git_repos\vascumap\bel_vascumap")))

import pandas as pd
from liffile import LifFile

from vascumap import VascuMap
from models import Pix2Pix, load_segmentation_model

W0401 15:39:43.562000 573084 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
c:\Users\taylorhearn\AppData\Local\miniconda3\envs\clean_vascumap\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.0)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
c:\Users\taylorhearn\AppData\Local\miniconda3\envs\clean_vascumap\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ── Helper functions ───────────────────────────────────────────────────────────

def discover_jobs(source_dir, force_mask=None, require_merged=True):
    """Find .lif/.tif/.tiff files and return (source, files, jobs) list."""
    source = Path(source_dir)
    if not source.is_dir():
        raise FileNotFoundError(f"source_dir does not exist: {source}")

    image_files = sorted(
        p for p in source.iterdir()
        if p.is_file() and p.suffix.lower() in (".lif", ".tif", ".tiff")
    )

    def _mask_mode(p, img_name=""):
        if force_mask is not None:
            return force_mask
        name = (p.name + " " + img_name).lower()
        if "marina" in name and "bead" in name:
            return "light"
        return "dark" if "marina" in name else False

    jobs = []
    for p in image_files:
        if p.suffix.lower() == ".lif":
            try:
                with LifFile(p) as lif:
                    for idx, img in enumerate(lif.images):
                        name = getattr(img, "name", "")
                        if require_merged and "merged" not in name.lower():
                            continue
                        jobs.append((p, idx, _mask_mode(p, name)))
            except Exception as exc:
                print(f"[SKIP] {p.name}: {exc}")
        else:
            if require_merged and "merged" not in p.name.lower():
                continue
            jobs.append((p, 0, _mask_mode(p)))

    return source, image_files, jobs


def run_batch(jobs, output_base, device_width_um, brightfield_channel=0,
              save_all_interim=False, model_p2p=None, model_unet=None,
              record_timings=False):
    """Run all jobs and optionally save a timing CSV."""
    results, timing_rows = [], []

    for i, (path, idx, mask_flag) in enumerate(jobs, 1):
        tag = f" (LIF #{idx})" if path.suffix.lower() == ".lif" else ""
        print(f"\n{'='*70}\n[{i}/{len(jobs)}] {path.name}{tag}  mask={mask_flag}\n{'='*70}")

        try:
            t0 = time.time()
            vm = VascuMap(
                use_device_segmentation_app=False,
                image_source_path=str(path),
                image_index=idx,
                device_width_um=device_width_um,
                mask_central_region=mask_flag,
                brightfield_channel=brightfield_channel,
                model_p2p=model_p2p,
                model_unet=model_unet,
            )
            vm.image_name = f"{path.stem}_img{idx}" if path.suffix.lower() == ".lif" else path.stem
            out_dir = Path(output_base) / vm.image_name
            print(f"  Output → {out_dir}")
            vm.pipeline(output_dir=out_dir, save_all_interim=save_all_interim)
            wall = time.time() - t0
            results.append((vm.image_name, "OK", ""))
            if record_timings:
                timing_rows.append({
                    "image_name": vm.image_name, "source_file": path.name,
                    "image_index": idx, "status": "OK",
                    "t_device_seg_s": round(getattr(vm, "_t_device_seg", 0), 1),
                    "t_preprocess_s": round(getattr(vm, "_t_preprocess", 0), 1),
                    "t_inference_s":  round(getattr(vm, "_t_inference", 0), 1),
                    "t_analysis_s":   round(getattr(vm, "_t_analysis", 0), 1),
                    "t_pipeline_s":   round(getattr(vm, "_t_total", 0), 1),
                    "t_job_wall_s":   round(wall, 1),
                })
        except Exception as exc:
            results.append((path.name + tag, "FAILED", str(exc)))
            if record_timings:
                timing_rows.append({
                    "image_name": path.name + tag, "source_file": path.name,
                    "image_index": idx, "status": "FAILED",
                })

        if record_timings and timing_rows:
            pd.DataFrame(timing_rows).to_csv(Path(output_base) / "batch_timings.csv", index=False)

    # Summary
    n_ok = sum(1 for _, s, _ in results if s == "OK")
    print(f"\n{'='*70}\nBatch complete: {n_ok}/{len(results)} succeeded")
    for name, status, msg in results:
        if status == "FAILED":
            print(f"  FAILED: {name}: {msg}")
    if record_timings and timing_rows:
        print(f"\nTimings → {Path(output_base) / 'batch_timings.csv'}")
        cols = ["image_name", "t_device_seg_s", "t_preprocess_s", "t_inference_s", "t_analysis_s", "t_pipeline_s"]
        print(pd.DataFrame(timing_rows).reindex(columns=cols).to_string(index=False))

    return results

In [3]:
shared_model_p2p = Pix2Pix(
    model_path=r"C:\Users\taylorhearn\git_repos\vascumap\luca_models\epoch=117-val_g_psnr=20.47-val_g_ssim=0.62.ckpt"
)
shared_model_unet = load_segmentation_model(
    r"C:\Users\taylorhearn\git_repos\vascumap\luca_models\best_full.pth"
)


In [4]:
# ── Configuration (edit here) ──────────────────────────────────────────────────
source_dir  = r"z:\Bel\Maria\New_Run"
output_base = r"Z:\Bel\Maria\New_Run_Outputs"

device_width_um       = 35.0
brightfield_channel   = 0
force_mask            = None     # "dark" / "light" / False / None (auto)
require_merged        = False
save_all_interim      = False
record_timings        = True

In [ ]:
# ── Discover jobs + run ────────────────────────────────────────────────────────
source, image_files, jobs = discover_jobs(source_dir, force_mask, require_merged)

print(f"Source: {source}\nFiles: {len(image_files)}  |  Jobs: {len(jobs)}")
for i, (p, idx, _) in enumerate(jobs, 1):
    tag = f" (LIF image {idx})" if p.suffix.lower() == ".lif" else ""
    print(f"  {i}. {p.name}{tag}")

run_batch(jobs, output_base, device_width_um, brightfield_channel,
          save_all_interim, shared_model_p2p, shared_model_unet, record_timings)

Source: z:\Bel\Maria\New_Run
Files: 3  |  Jobs: 22
  1. Exp10_20260318.lif (LIF image 0)
  2. Exp10_20260318.lif (LIF image 1)
  3. Exp10_20260318.lif (LIF image 2)
  4. Exp10_20260318.lif (LIF image 3)
  5. Exp10_20260318.lif (LIF image 4)
  6. Exp10_20260325.lif (LIF image 0)
  7. Exp10_20260325.lif (LIF image 1)
  8. Exp10_20260325.lif (LIF image 2)
  9. Exp10_20260325.lif (LIF image 3)
  10. Exp10_20260325.lif (LIF image 4)
  11. Exp10_20260325.lif (LIF image 5)
  12. Exp10_20260325.lif (LIF image 6)
  13. Exp10_20260331.lif (LIF image 0)
  14. Exp10_20260331.lif (LIF image 1)
  15. Exp10_20260331.lif (LIF image 2)
  16. Exp10_20260331.lif (LIF image 3)
  17. Exp10_20260331.lif (LIF image 4)
  18. Exp10_20260331.lif (LIF image 5)
  19. Exp10_20260331.lif (LIF image 6)
  20. Exp10_20260331.lif (LIF image 7)
  21. Exp10_20260331.lif (LIF image 8)
  22. Exp10_20260331.lif (LIF image 9)

[1/22] Exp10_20260318.lif (LIF #0)  mask=False
  [LIF] Raw array shape: (31, 6362, 3778)  dtype=uin

100%|██████████| 80/80 [00:16<00:00,  4.96it/s]
